# Workshop 2 — Blur an Image on a GPU 🖼️➡️🌫️

Last time you built a tool on a Linux machine. Today you run **your own code on a real GPU** — the same kind of chip that powers ChatGPT.

**Two things before you start:**

1. Turn on the GPU: **Runtime → Change runtime type → T4 GPU → Save.**
2. Then just run each cell in order with **Shift+Enter**. That's it.


## 1. Get everything (one click)
This grabs the whole workshop — the image and all the files — straight from GitHub. Nothing to download or upload.

In [ ]:
%cd /content
!git clone https://github.com/daryl-888/Workshop2.git 2>/dev/null || echo 'already downloaded — continuing'
%cd /content/Workshop2
!ls


## 2. Check you have a GPU
You should see a table mentioning **Tesla T4**. If it errors, go to **Runtime → Change runtime type → T4 GPU** and run this again.

In [ ]:
!nvidia-smi

## 3. Fill in the blanks, then write the program
Below is **your CUDA program with 9 blanks** (`/* TODO n: ... */`).

1. Read the comment above each blank — it says exactly what to write.
2. Replace each `/* TODO n: ... */` with real code.
3. Run this cell — the `%%writefile` line saves your edited code to `blur.cu`.

It won't compile until every blank is filled — that's on purpose. Hints are in the repo README.

In [ ]:
%%writefile blur.cu
// ============================================================
//  blur_template.cu  —  Box blur on the GPU (FILL IN THE BLANKS)
//
//  One CUDA thread blurs one pixel by averaging its neighbours.
//  Your job: fill in each  /* TODO n: ... */  blank. There are 9.
//  The file will NOT compile until every blank is filled — that's
//  intentional. Work top to bottom; the comments tell you what
//  each line should compute. Stuck? See HINTS in the README, and
//  blur_solution.cu has the full answer if you're truly stuck.
//
//  Build & run (Colab or any GPU + CUDA machine):
//      nvcc blur_template.cu -o blur
//      ./blur                    # reads sample_1920x1280.ppm, writes output.ppm
// ============================================================
#include <cstdio>
#include <cstdlib>

#define BLUR_SIZE 3          // radius; a (2*3+1) x (2*3+1) = 7x7 window

// ---- The kernel: this code runs on the GPU, once per thread ----
__global__
void blurKernel(unsigned char *out, unsigned char *in, int w, int h) {
    // Which pixel does THIS thread own? Combine the block index, the
    // block dimension, and the thread index for BOTH x (col) and y (row).
    int col = /* TODO 1: block index x * block dim x + thread index x */;
    int row = /* TODO 2: block index y * block dim y + thread index y */;

    // The grid is rounded up, so some threads land outside the image.
    // Only work if this pixel is actually inside the width and height.
    if (/* TODO 3: col in range AND row in range */) {
        int rVal = 0, gVal = 0, bVal = 0;
        int pixels = 0;

        // Walk the square neighbourhood around (row, col).
        for (int blurRow = -BLUR_SIZE; blurRow <= BLUR_SIZE; ++blurRow) {
            for (int blurCol = -BLUR_SIZE; blurCol <= BLUR_SIZE; ++blurCol) {
                int curRow = row + blurRow;
                int curCol = col + blurCol;

                if (curRow >= 0 && curRow < h && curCol >= 0 && curCol < w) {
                    // Flat row-major index into the RGB array (3 bytes per pixel).
                    int idx = /* TODO 4: (curRow * w + curCol) * 3 */;
                    rVal += in[idx + 0];
                    gVal += in[idx + 1];
                    bVal += in[idx + 2];
                    ++pixels;
                }
            }
        }

        // Average = sum / number of neighbours counted. Write all 3 channels.
        int outIdx = (row * w + col) * 3;
        out[outIdx + 0] = /* TODO 5a: average of rVal over pixels, as unsigned char */;
        out[outIdx + 1] = /* TODO 5b: average of gVal over pixels, as unsigned char */;
        out[outIdx + 2] = /* TODO 5c: average of bVal over pixels, as unsigned char */;
    }
}

// ---- Host helper: move data to the GPU, launch, bring it back ----
void blurOnGPU(unsigned char *in_h, unsigned char *out_h, int width, int height, int size) {
    unsigned char *in_d, *out_d;

    // 1. Allocate memory ON the GPU (device).
    cudaMalloc((void **)&in_d,  size);
    cudaMalloc((void **)&out_d, size);

    // 2. Copy the input image from CPU (host) -> GPU (device).
    cudaMemcpy(in_d, in_h, size, /* TODO 6: copy direction, host -> device */);

    // 3. Choose the grid. Block = 16x16 threads. The grid must have ENOUGH
    //    blocks to cover every pixel, rounding UP so nothing is missed.
    dim3 dimBlock(16, 16, 1);
    dim3 dimGrid(/* TODO 7a: blocks across width */, /* TODO 7b: blocks down height */, 1);

    // 4. Launch the kernel with your grid and block.
    blurKernel<<</* TODO 8: dimGrid, dimBlock */>>>(out_d, in_d, width, height);

    // 5. Copy the finished image back GPU (device) -> CPU (host).
    cudaMemcpy(out_h, out_d, size, /* TODO 9: copy direction, device -> host */);

    // 6. Free the GPU memory.
    cudaFree(in_d);
    cudaFree(out_d);
}

// ---- Read a binary (P6) PPM image from disk (given) ----
void ReadImage(unsigned char **imgData, int *width, int *height) {
    FILE *f = fopen("images/sample_1920x1280.ppm", "rb");
    if (!f) { printf("Could not open sample_1920x1280.ppm\n"); exit(EXIT_FAILURE); }
    fscanf(f, "P6\n%d %d\n255\n", width, height);
    *imgData = (unsigned char *)malloc((*width) * (*height) * 3);
    fread(*imgData, 1, (*width) * (*height) * 3, f);
    fclose(f);
}

// ---- Write a binary (P6) PPM image to disk (given) ----
void WriteImage(unsigned char *pixOut, int width, int height) {
    FILE *fptr = fopen("output.ppm", "wb");
    if (!fptr) { printf("Could not open output.ppm for writing\n"); return; }
    fprintf(fptr, "P6\n%d %d\n255\n", width, height);
    fwrite(pixOut, 1, width * height * 3, fptr);
    fclose(fptr);
    printf("Wrote output.ppm (%dx%d)\n", width, height);
}

int main() {
    unsigned char *imgData, *pixOut;
    int width, height;

    ReadImage(&imgData, &width, &height);

    int size = width * height * 3;
    pixOut = (unsigned char *)malloc(size);

    blurOnGPU(imgData, pixOut, width, height, size);

    WriteImage(pixOut, width, height);

    free(imgData);
    free(pixOut);
    return 0;
}


## 4. Compile with `nvcc`
`nvcc` is the CUDA compiler. No errors = your blanks are valid CUDA. An error points at the line to fix.

In [ ]:
!nvcc blur.cu -o blur

## 5. Run it on the GPU
Reads the image, blurs it on the GPU, writes `output.ppm`.

In [ ]:
!./blur

## 6. See your result 🎉
Left: original. Right: your GPU-blurred image. If the right side looks softer, **you just ran code on a GPU.**

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
before = Image.open('images/sample_1920x1280.ppm')
after  = Image.open('output.ppm')
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(before); ax[0].set_title('Before'); ax[0].axis('off')
ax[1].imshow(after);  ax[1].set_title('After — blurred on the GPU'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## Done!
You wrote the kernel that does this — one thread per pixel, thousands running in parallel.

**Experiment:** go back to cell 3, change `#define BLUR_SIZE 3` to `7` or `15`, then re-run cells 3–6. Bigger number = stronger blur (each thread averages a bigger neighbourhood).

**The big idea (see the slides):** "one thread per output element" is exactly how GPUs do the *matrix multiplies* inside every LLM. You just did the same thing that makes ChatGPT fast — on a picture.